# Men's Tournament Logistic Regression

This notebook fits a logistic regression model using `diff_massey_rating` and `diff_TeamAvgScore`, uses `GridSearchCV` to choose the L2 regularization strength based on Brier score, and evaluates the tuning season-by-season with leave-one-season-out cross-validation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import GridSearchCV, GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [2]:
DATA_DIR = Path.cwd()
FEATURES = ["diff_massey_rating", "diff_TeamAvgScore"]
TARGET = "Team1Win"

mtrain = pd.read_csv(DATA_DIR / "mtrain2026.csv")
mtest = pd.read_csv(DATA_DIR / "mtest2026.csv")

X_train = mtrain[FEATURES]
y_train = mtrain[TARGET]
X_test = mtest[FEATURES]
season_groups = mtrain["ID"].str.split("_").str[0]

print(f"Training seasons: {sorted(season_groups.unique())}")


Training seasons: ['2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2021', '2022', '2023', '2024']


In [3]:
preprocessor = StandardScaler()
season_cv = GroupKFold(n_splits=season_groups.nunique())

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                penalty="l2",
                solver="saga",
                max_iter=3000,
                random_state=1,
            ),
        ),
    ]
)

param_grid = {
    "classifier__C": np.logspace(-3, 2, 10),
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=season_cv,
    scoring="neg_brier_score",
    n_jobs=1,
    verbose=1,
    refit=True,
)


In [4]:
grid_search.fit(X_train, y_train, groups=season_groups)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_c = best_params["classifier__C"]
best_cv_brier = -grid_search.best_score_

cv_report = (
    pd.DataFrame(grid_search.cv_results_)
    [["param_classifier__C", "mean_test_score", "std_test_score", "rank_test_score"]]
    .rename(
        columns={
            "param_classifier__C": "C",
            "mean_test_score": "mean_cv_brier",
            "std_test_score": "std_cv_brier",
            "rank_test_score": "rank",
        }
    )
)
cv_report["mean_cv_brier"] = -cv_report["mean_cv_brier"]
cv_report = cv_report.sort_values("rank").reset_index(drop=True)

cv_brier = -cross_val_score(
    best_model,
    X_train,
    y_train,
    cv=season_cv,
    groups=season_groups,
    scoring="neg_brier_score",
)
train_prob = best_model.predict_proba(X_train)[:, 1]

print("Best hyperparameters:")
for name, value in best_params.items():
    print(f"  {name} = {value}")

print(f"Best mean season-by-season CV Brier score: {best_cv_brier:.4f}")
print(f"Mean season-by-season CV Brier score: {cv_brier.mean():.4f} (+/- {cv_brier.std():.4f})")
print(f"Training Brier score: {brier_score_loss(y_train, train_prob):.4f}")

display(cv_report)

coef_table = pd.DataFrame(
    {
        "feature": FEATURES,
        "coefficient": best_model.named_steps["classifier"].coef_[0],
    }
).sort_values("coefficient", key=np.abs, ascending=False)
display(coef_table)

mens_predictions = mtest[["ID", "Team1Name", "Team2Name"]].copy()
mens_predictions["Team1WinProb"] = best_model.predict_proba(X_test)[:, 1]
display(mens_predictions.head(10))

output_path = DATA_DIR / "mtest2026_predictions_logreg.csv"
mens_predictions.to_csv(output_path, index=False)
print(f"Saved predictions to {output_path.name}")


Fitting 14 folds for each of 10 candidates, totalling 140 fits


/Users/mcmcclur/miniforge3/envs/hf/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mcmcclur/miniforge3/envs/hf/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mcmcclur/miniforge3/envs/hf/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and w

Best hyperparameters:
  classifier__C = 0.5994842503189409
Best mean season-by-season CV Brier score: 0.1963
Mean season-by-season CV Brier score: 0.1963 (+/- 0.0183)
Training Brier score: 0.1955


/Users/mcmcclur/miniforge3/envs/hf/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mcmcclur/miniforge3/envs/hf/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mcmcclur/miniforge3/envs/hf/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and w

,C,mean_cv_brier,std_cv_brier,rank
0,0.599484,0.196267,0.018296,1
1,2.154435,0.196270,0.018407,2
2,7.742637,0.196272,0.018438,3
3,27.825594,0.196272,0.018447,4
4,100.000000,0.196272,0.018449,5
5,0.166810,0.196277,0.017920,6
6,0.046416,0.196514,0.016776,7
7,0.012915,0.198440,0.014126,8
8,0.003594,0.206252,0.010096,9
9,0.001000,0.222263,0.005770,10


,feature,coefficient
0,diff_massey_rating,1.271726
1,diff_TeamAvgScore,-0.076794


,ID,Team1Name,Team2Name,Team1WinProb
0,2025_1103_1104,Akron,Alabama,0.084895
1,2025_1103_1106,Akron,Alabama St,0.780769
2,2025_1103_1110,Akron,American Univ,0.726516
3,2025_1103_1112,Akron,Arizona,0.102564
4,2025_1103_1116,Akron,Arkansas,0.180156
5,2025_1103_1120,Akron,Auburn,0.054535
6,2025_1103_1124,Akron,Baylor,0.138041
7,2025_1103_1136,Akron,Bryant,0.631503
8,2025_1103_1140,Akron,BYU,0.140126
9,2025_1103_1155,Akron,Clemson,0.151530


Saved predictions to mtest2026_predictions_logreg.csv
